# Electronegativity Equalization Method

This notebook shows how to compute atomic charges using the electronegativity equalization method. The basic idea is that dependence of the energy of an atom (or group) in a molecule can be written as a Taylor series,

$$
E_A(N_0 + \Delta N) = E(N_0) + \left( \frac{\partial E_A}{\partial N} \right)_{N_0} \Delta N + \frac{1}{2} \left( \frac{\partial^2 E_A}{\partial N^2} \right)_{N_0} (\Delta N)^2 + \ldots
$$

which, by convention, we truncate at second order. The first derivative is the electronic chemical potential (the negative of the electronegativity, $\chi$; also called the Fermi level) and the second derivative is the chemical hardness, $\eta$ (also called the band gap). Thus, we can write

$$
E_A(N_0 + \Delta N) = E(N_0) + \mu_A \Delta N + \tfrac{1}{2} \eta_A (\Delta N)^2
$$

Using the known energies of the $N_0$, $N_0 + 1$, and $N_0 - 1$ states, we can solve for $\mu_A$ and $\eta_A$, obtaining the Mulliken electronegativity and Parr-Pearson hardness,
$$
\mu_A = \frac{1}{2} (E(N_0 + 1) - E(N_0 - 1)) = -\tfrac{1}{2} (I + A)
$$
$$
\eta_A = E(N_0 + 1) + E(N_0 - 1) - 2E(N_0) = I - A
$$
where $I = E(N_0) - E(N_0 + 1) \ge 0$ is the ionization potential and $A = E(N_0 - 1) - E(N_0) \ge 0$ is the electron affinity. 

The total energy is then,
$$
\begin{split}
E_{\text{total}} &= \sum_A E_A(N_{A,0} + \Delta N_A) \\
&= \sum_A E_A(N_{A,0}) + \sum_A \mu_A \Delta N_A + \frac{1}{2} \sum_A \eta_A (\Delta N_A)^2 + \frac{1}{2} \sum_{A \ne B} J_{AB} \Delta N_A \Delta N_B
\end{split}
$$
This assumes that the atoms/groups "reference state" is neutral so that their charge is just $q_A = -\Delta N_A$. The last term is the Coulomb interaction between the charges, where $J_{AB}$ is the Coulomb integral between the charge distributions of atoms/groups $A$ and $B$. We could evaluate the Coulomb integral between the charge distributions analytically using [`AtomDB`](https://atomdb.qcdevs.org/api/index.html) or [`Bfit`](https://bfit.qcdevs.org/), but this is not what is usually done in practice. Instead, the Coulomb interaction is usually screened, so that for electrostatic interactions between bonded atoms and atoms bonded to a common atom (linked by a bond angle) the Coulomb interaction is nearly neglected, while for atoms linked by a torsion the electrostatic interaction is screened. We'll use an [erfgau screening](https://arxiv.org/pdf/physics/0410062),
$$
J_{AB} = \frac{\operatorname{erf}(\alpha R_{AB})}{R_{AB}} - \frac{2\alpha}{\sqrt{\pi}}  \exp\left({-\tfrac{\alpha^2R_{AB}^2}{3}}\right)
$$

Now, to obtain the charges, we minimize the total energy with respect to the amount of electron transfer, $\Delta N_A$, subject to the constraint that the total charge is fixed, $\sum_A \Delta N_A = -Q$. 
$$
\min_{\{\Delta N_A\}} E_{\text{total}} \quad \text{subject to} \quad \sum_A \Delta N_A = -Q
$$
Forcing the constraint with the Lagrange multiplier, $\mu_{\text{total}}$, we have the Lagrangian,
$$
\mathcal{L} = E_{\text{total}} + \mu_{\text{total}} \left( \sum_A \Delta N_A + Q \right)
$$
Differentiating the Lagrangian and setting it equal to zero gives a system of $P+1$ linear equations, where $P$ is the number of atoms/groups,
$$
\begin{split}
0 &= \frac{\partial E_{\text{total}}}{\partial \Delta N_1} + \mu_{\text{total}} \\
0 &= \frac{\partial E_{\text{total}}}{\partial \Delta N_2} + \mu_{\text{total}} \\
&\vdots \\
0 &= \frac{\partial E_{\text{total}}}{\partial \Delta N_P} + \mu_{\text{total}} \\
0 &= \sum_A \Delta N_A + Q
\end{split}
$$
Using the explicit expression for the total energy, this system of equations can be written in matrix form as
$$
\mathbf{A}\boldsymbol{\delta}=\mathbf{m},
$$
where
$$
\begin{aligned}
\mathbf{A} &=
\begin{bmatrix}
\eta_1  & J_{12} & \cdots & J_{1P} & 1 \\
J_{21}  & \eta_2 & \cdots & J_{2P} & 1 \\
\vdots  & \vdots & \ddots & \vdots & \vdots \\
J_{P1}  & J_{P2} & \cdots & \eta_P & 1 \\
1       & 1      & \cdots & 1      & 0
\end{bmatrix}, \\[1em]
\boldsymbol{\delta} &=
\begin{bmatrix}
\Delta N_1 \\
\Delta N_2 \\
\vdots \\
\Delta N_P \\
\mu_{\mathrm{total}}
\end{bmatrix}, \\[1em]
\mathbf{m} &=
\begin{bmatrix}
\mu_1 \\
\mu_2 \\
\vdots \\
\mu_P \\
- Q
\end{bmatrix}.
\end{aligned}
$$

We take benchmark values of the chemical potentials and hardnesses from [this reference](https://pubs.rsc.org/en/content/articlelanding/2016/cp/c6cp04533b), using [`AtomDB`](https://atomdb.qcdevs.org/api/index.html).







In [ ]:
!pip install qc-atomdb
!pip install qc-iodata

In [5]:
import numpy as np
from scipy.special import erf

def erfgau(r, alpha=0.5):
    """
    Savin's erfgau long-range interaction:

        v(r) = erf(mu*r)/r - (2*mu/sqrt(pi)) * exp(-(mu*r)^2 / 3)

    Parameters
    ----------
    r
        Interparticle distance(s). May be a positive float or a NumPy array
        of positive distances.
    alpha
        Range-separation parameter. Must be nonnegative. The default
        value is set so that for a 1-3 interactions of ~6 bohr the erfgau
        screening is roughly 20%. As this number increases the screening becomes
        less aggressive.

    Returns
    -------
    float or np.ndarray
        The erfgau interaction evaluated at r.
    """
    if alpha < 0.0:
        raise ValueError("alpha must be nonnegative.")

    # Set a tolerance for zero to avoid issues with very small distances
    tol = 1e-12

    # Treat scalar case separately
    if np.isscalar(r):
        r = float(r)
        if r <= 0:
            raise ValueError("r must be positive.")
        elif r < tol:
            # For very small r, use the limit as r -> 0 to avoid numerical issues
            return 0.0
        else:
            x = alpha * r
            return erf(x) / r - (2.0 * alpha / np.sqrt(np.pi)) * np.exp(-(x**2) / 3.0)

    r_arr = np.asarray(r, dtype=float)
    if np.any(r_arr <= 0.0):
        raise ValueError("All elements of r must be positive.")

    x = alpha * r_arr

    # Form an array to hold the input and initalize it to zero
    erfgau = np.zeros_like(r_arr)
    # For very small r, zero is the answer so the intialization is correct.
    # For larger r, compute the erfgau interaction as normal. We can do this with
    # np.where.
    erfgau = np.where(r_arr < tol, 0.0, erf(x) / r_arr - (2.0 * alpha / np.sqrt(np.pi)) * np.exp(-(x**2) / 3.0))

    return erfgau

In [14]:
# Load the Water molecule geometry from an XYZ file (using IOData)
from urllib.request import urlretrieve
from iodata import load_one

url = "https://github.com/theochem/chemtools/raw/refs/heads/master/chemtools/data/examples/dichloropyridine26_q+0.fchk"
urlretrieve(url, "dichloropyridine26_q0.fchk")

mol0 = load_one("dichloropyridine26_q0.fchk")

print(mol0.atnums)
print(mol0.atcoords)

[17 17  7  6  6  6  6  6  1  1  1]
[[-4.95130924e+00  2.31173977e+00 -2.64561659e-04]
 [ 4.95149821e+00  2.31139962e+00  2.07869875e-04]
 [ 7.55890453e-05  1.84250188e+00 -1.88972613e-05]
 [-9.44863066e-05 -3.41775868e+00  1.13383568e-04]
 [-2.28277027e+00 -2.09580077e+00 -1.88972613e-05]
 [ 2.28267578e+00 -2.09595195e+00  1.70075352e-04]
 [-2.13763930e+00  5.39856962e-01 -3.77945227e-05]
 [ 2.13769600e+00  5.39705784e-01  7.55890453e-05]
 [-1.70075352e-04 -5.47831606e+00  1.51178091e-04]
 [-4.10063012e+00 -3.05289036e+00 -7.55890453e-05]
 [ 4.10047894e+00 -3.05315492e+00  2.83458920e-04]]


In [ ]:
# Retrieve the chemical potentials and hardnesses using AtomDB
import atomdb
from atomdb import Element

def nist_mu_eta_for_atoms(atnums):
    """return a numpy array of mu's and eta's for the given atomic numbers"""
    atnums = np.asarray(atnums, dtype=int)
    mu = np.empty(len(atnums))
    eta = np.empty(len(atnums))
    for i, z in enumerate(atnums):
        mult = Element(z).mult          # neutral ground-state multiplicity
        sp = atomdb.load(z, charge=0, mult=mult, dataset="nist")
        mu[i] = sp.mu
        eta[i] = sp.eta
    return mu, eta

# Example with your molecule atomic numbers:
# atnums = h2o.atnums   # or your molecule's atnums
mu, eta = nist_mu_eta_for_atoms(mol0.atnums)
print("mu =", mu)
print("eta =", eta)


mu = [-0.30465188 -0.30465188 -0.26716757 -0.23005076 -0.23005076 -0.23005076
 -0.23005076 -0.23005076 -0.26386013 -0.26386013 -0.26386013]
eta = [0.34360616 0.34360616 0.53396765 0.36749322 0.36749322 0.36749322
 0.36749322 0.36749322 0.4718613  0.4718613  0.4718613 ]


In [ ]:
# Set up the equations to solve for the charges and print out the charges.  